# DealFinder Part 10 — Fine-Tune the Extractor with QLoRA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudcodetree/tutorial-dealfinder/blob/main/notebooks/finetune_qlora.ipynb)

Companion notebook for [Part 10 of the DealFinder course](https://cloudcodetree.com/tutorials/dealfinder-finetune). It fine-tunes `Llama-3.2-1B-Instruct` with QLoRA so the extractor stops attributing an Anker headphone sold by `mountainlifestyle.ca` to the marketplace instead of the manufacturer.

**Requirements:** a CUDA GPU — the free Colab T4 is enough (Runtime → Change runtime type → T4 GPU). Training ~220 examples for 3 epochs takes ≈25 minutes.

**Honesty note (matches the lesson):** the lesson's metric table (29% → 4% retailer bleed) is *illustrative*. This notebook builds its labels with a weak-labeling heuristic instead of the hand-labeled gold CSV, so your exact numbers will differ run to run and from the table — the *shape* of the improvement is what to look for.

## 1 · GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No CUDA GPU. In Colab: Runtime -> Change runtime type -> T4 GPU. "
    "This notebook (QLoRA + bitsandbytes) cannot run on CPU."
)
print(torch.cuda.get_device_name(0))

## 2 · Install training deps (~2 min)

In [ ]:
%pip install -q -U transformers peft trl bitsandbytes datasets accelerate

## 3 · Fetch the real snapshot

The same frozen 270-item `electronics-2026-07.json` every course number is pinned to.

In [ ]:
import json, urllib.request

SNAPSHOT_URL = (
    "https://raw.githubusercontent.com/cloudcodetree/tutorial-dealfinder/"
    "main/data/snapshots/electronics-2026-07.json"
)
raw = json.loads(urllib.request.urlopen(SNAPSHOT_URL).read())
items = raw["items"] if isinstance(raw, dict) and "items" in raw else raw
print(len(items), "items")

retailer_tokens = {"walmart", "target", "costco", "macy", "best buy", "amazon", "mountainlifestyle", "kohl"}
polluted = [i for i in items if any(t in (i.get("brand") or "").lower() for t in retailer_tokens)]
print(f"{len(polluted)} of {len(items)} rows have a retailer in 'brand'")  # expect 154 of 270

## 4 · Build the chat-format dataset (weak labels)

The lesson's `build_dataset.py` reads a hand-labeled gold CSV. Here we stand in for it with a
weak-labeling heuristic: strip retailer prefixes, then take the manufacturer from a known-brand
list scan of the title. Inspect the sample below — weak labels are imperfect by design, and the
titles the heuristic gets right are exactly the supervision the model needs to stop copying
retailer tokens.

In [ ]:
import random, re

SYSTEM_PROMPT = (
    "You are a product-listing parser. Given an electronics title, "
    "return JSON with keys manufacturer (string|null), model (string|null), "
    "condition (new|refurb|used|null). Return ONLY valid JSON."
)

KNOWN_BRANDS = [
    "Sony", "Bose", "Anker", "Soundcore", "JBL", "Beats", "Apple", "Samsung",
    "Sennheiser", "COWIN", "Skullcandy", "Logitech", "Razer", "Corsair", "HP",
    "Dell", "Lenovo", "ASUS", "Acer", "LG", "TCL", "Hisense", "Roku", "Amazon",
    "Kindle", "SanDisk", "Seagate", "Crucial", "Kingston", "TP-Link", "Netgear",
    "Garmin", "Fitbit", "JLab", "Belkin", "Onn", "Insignia", "Sceptre", "eufy",
]

def weak_label(title: str) -> dict | None:
    t = title
    # strip 'Retailer - ' style prefixes and domain prefixes
    t = re.sub(r"^[A-Za-z&'. ]{2,20} - ", "", t)
    t = re.sub(r"^\S+\.(com|ca|net) ", "", t)
    manufacturer = next((b for b in KNOWN_BRANDS if re.search(rf"\\b{re.escape(b)}\\b", t, re.I)), None)
    if manufacturer is None:
        return None  # unlabelable by the heuristic -> excluded from training
    low = title.lower()
    condition = "refurb" if ("refurb" in low or "renewed" in low) else "used" if "pre-owned" in low else "new"
    m = re.search(rf"{re.escape(manufacturer)}\\s+([A-Za-z0-9][A-Za-z0-9-]*(?:\\s[A-Z0-9][A-Za-z0-9-]*)?)", t, re.I)
    return {"manufacturer": manufacturer, "model": m.group(1).strip() if m else None, "condition": condition}

def to_chat_pair(title: str, label: dict) -> dict:
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": title},
        {"role": "assistant", "content": json.dumps(label)},
    ]}

labeled = [(i["title"], weak_label(i["title"])) for i in items]
labeled = [(t, l) for t, l in labeled if l is not None]

random.seed(42)
random.shuffle(labeled)
split = int(len(labeled) * 0.8)
train_pairs = [to_chat_pair(t, l) for t, l in labeled[:split]]
eval_pairs  = [to_chat_pair(t, l) for t, l in labeled[split:]]
print(f"train {len(train_pairs)} · eval {len(eval_pairs)}")
for p in train_pairs[:3]:
    print(p["messages"][1]["content"][:70], "->", p["messages"][2]["content"])

## 5 · Load the 4-bit base + LoRA config (the lesson's exact hyperparameters)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType

BASE_MODEL = "unsloth/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

## 6 · Train (~25 min on a T4)

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="adapters/extractor-v1/",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    warmup_steps=10,
    save_strategy="epoch",
    logging_steps=5,
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=Dataset.from_list(train_pairs),
    peft_config=lora_config,
)
trainer.train()

## 7 · The payoff — hero-cast hard titles, before vs. after

In [ ]:
HARD_TITLES = [
    "Walmart - COWIN E7 Active Noise Cancelling Headphones Bluetooth",
    "mountainlifestyle.ca Anker Soundcore Q20i Wireless Over-Ear Headphones",
    "Sony WH-1000XM5 Wireless Industry Leading Noise Canceling Headphones",
]

def generate(m, title: str) -> str:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": title}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(m.device)
    out = m.generate(inputs, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

tuned = trainer.model
for t in HARD_TITLES:
    print(t)
    with tuned.disable_adapter():
        print("  base   :", generate(tuned, t))
    print("  adapter:", generate(tuned, t))
    print()

## 8 · Score the held-out split — manufacturer accuracy & retailer bleed

In [ ]:
RETAILERS = {"walmart", "target", "costco", "macy's", "macy", "best buy", "kohl's", "mountainlifestyle.ca", "amazon"}

def parse_json(s: str) -> dict:
    try:
        return json.loads(re.search(r"\\{.*\\}", s, re.S).group(0))
    except Exception:
        return {}

def score(use_adapter: bool) -> dict:
    manu_hits = bleed = 0
    for pair in eval_pairs:
        title = pair["messages"][1]["content"]
        gold = json.loads(pair["messages"][2]["content"])
        if use_adapter:
            out = parse_json(generate(tuned, title))
        else:
            with tuned.disable_adapter():
                out = parse_json(generate(tuned, title))
        pred = (out.get("manufacturer") or "").lower()
        manu_hits += pred == (gold["manufacturer"] or "").lower()
        bleed += pred in RETAILERS
    n = len(eval_pairs)
    return {"manufacturer_accuracy": manu_hits / n, "retailer_bleed": bleed / n}

print("base   :", score(use_adapter=False))
print("adapter:", score(use_adapter=True))

## 9 · Save the adapter

In Colab, download the zip via the file browser (folder icon, left sidebar). The lesson's
`llm_extract(..., adapter_path=...)` extension is where this artifact slots into the pipeline —
and merging (`merge_and_unload()`) produces the single self-contained model file the lesson's
inference path uses.

In [ ]:
trainer.model.save_pretrained("adapters/extractor-v1")
tokenizer.save_pretrained("adapters/extractor-v1")
!zip -qr extractor-v1-adapter.zip adapters/extractor-v1
print("saved adapters/extractor-v1 + extractor-v1-adapter.zip")